# Day 23 — Final Validation & Performance Comparison

## Objective

The objective of Day 23 is to perform final validation of the frozen project configurations on held-out data,measure inference performance and model size,and compare the final methods against the Original YOLOv8n baseline.

The final configurations are evaluated without changing the frozen preprocessing,model settings,or dataset strategy. No retraining or tuning is performed in Day 23.

## Final Frozen Configurations

### DIATAquarium

Original Image + YOLOv8n

No local test folder exists,so final validation is performed on the validation set.

### Aquatic Plant

Original Image + White Balance + CLAHE + YOLOv8n (H03)

Learning Rate: 0.0005
Batch Size: 32
Epochs: 30
Image Size: 640
Optimizer: AdamW

### Well

Original Image + White Balance + CLAHE + YOLOv8n (H04)

Learning Rate: 0.001
Batch Size: 16
Epochs: 30
Image Size: 640
Optimizer: AdamW

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 10.7 MB/s eta 0:00:00


## Note on Baseline Comparison

The baseline figures used in Day 23 for Aquatic Plant and Well are their test-set baseline results obtained in the earlier baseline experiments.

These differ from the validation-set baseline figures used during candidate selection in Day 19 and Day 21 because validation and test sets contain different images.

Therefore, Day 19/21 results are used for validation-based model selection,while Day 23 results are used for final test-set comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from ultralytics import YOLO
import pandas as pd

results_log=[]

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
import shutil
import os
source="/content/drive/MyDrive/Dataset_V1"
destination="/content/Dataset_V1"
if os.path.exists(destination):
    print("Dataset_V1 already exists in /content")
else:
    print("Copying Dataset_V1 to /content...")
    shutil.copytree(source,destination)
    print("Dataset_V1 copied successfully.")
print("\nDatasets:")
print(os.listdir(destination))

Copying Dataset_V1 to /content...
Dataset_V1 copied successfully.

Datasets:
['well.v8i.yolov8', 'DIATAquarium.v4i.yolov8', 'split_plan', 'preprocessing_config.txt', 'Aquatic Plant.v2i.yolov8']


In [ ]:
# DIAT - Final validation on validation set
diat_model=YOLO("/content/drive/MyDrive/Day22_Final_Model/DIAT_Baseline/weights/best.pt")
diat_val=diat_model.val(
    data="/content/Dataset_V1/DIATAquarium.v4i.yolov8/baseline.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0
)
diat_result={
    "Dataset":"DIAT",
    "Configuration":"Original + YOLOv8n",
    "Split":"Validation",
    "Precision":round(diat_val.box.mp*100,2),
    "Recall":round(diat_val.box.mr*100,2),
    "mAP50":round(diat_val.box.map50*100,2),
    "mAP50_95":round(diat_val.box.map*100,2)
}
print(diat_result)

Ultralytics 8.4.158 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 3.6±4.8 ms, read: 17.9±10.9 MB/s, size: 152.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/drive/MyDrive/Dataset_V1/DIATAquarium.v4i.yolov8/valid/labels.cache... 773 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 773/773 108.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 2.8it/s 17.3s
                   all        773       1504        0.9      0.921      0.927      0.659
         Aquatic Plant        358        409       0.98      0.988      0.986      0.791
                Camera         15         15      0.969      0.933      0.935      0.572
           Conch-Shell       

In [ ]:
diat_result={
    "Dataset":"DIAT",
    "Configuration":"Original + YOLOv8n",
    "Split":"Validation",
    "Precision":90.02,
    "Recall":92.09,
    "mAP50":92.73,
    "mAP50_95":65.92
}
print(diat_result)

{'Dataset': 'DIAT', 'Configuration': 'Original + YOLOv8n', 'Split': 'Validation', 'Precision': 90.02, 'Recall': 92.09, 'mAP50': 92.73, 'mAP50_95': 65.92}


In [ ]:
import os
enhanced_drive="/content/drive/MyDrive/Dataset_V1_Enhanced"
print("Dataset_V1_Enhanced in Google Drive:",os.path.exists(enhanced_drive))
if os.path.exists(enhanced_drive):
    print("\nContents:")
    print(os.listdir(enhanced_drive))

Dataset_V1_Enhanced in Google Drive: True

Contents:
['DIATAquarium.v4i.yolov8', 'Aquatic Plant.v2i.yolov8', 'well.v8i.yolov8']


In [ ]:
import os
import shutil

source="/content/drive/MyDrive/Dataset_V1_Enhanced"
destination="/content/Dataset_V1_Enhanced"

if os.path.exists(destination):
    shutil.rmtree(destination)

shutil.copytree(source,destination)

print("Dataset_V1_Enhanced copied successfully.")
for dataset in os.listdir(destination):
    path=os.path.join(destination,dataset)
    if os.path.isdir(path):
        print(dataset,":",os.listdir(path))

Dataset_V1_Enhanced copied successfully.
well.v8i.yolov8 : ['train', 'valid', 'test', 'data.yaml']
DIATAquarium.v4i.yolov8 : ['enhanced.yaml', 'train', 'valid', 'data.yaml']
Aquatic Plant.v2i.yolov8 : ['train', 'valid', 'H03.yaml', 'test', 'data.yaml']


In [ ]:
import os
yaml_files=[
    "/content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml",
    "/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H03.yaml",
    "/content/Dataset_V1_Enhanced/well.v8i.yolov8/data.yaml"
]
for path in yaml_files:
    print("\nFile:",path)
    print("Exists:",os.path.exists(path))

    if os.path.exists(path):
        with open(path,"r") as f:
            print(f.read())


File: /content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8/enhanced.yaml
Exists: True
path: /content/drive/MyDrive/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8
train: train/images
val: valid/images
nc: 11
names:
  0: Aquatic Plant
  1: Camera
  2: Conch-Shell
  3: Fish
  4: Fishes
  5: Pluco
  6: Shark
  7: Temperature Sensor
  8: Water-Pump
  9: Water-filter
  10: betta


File: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/H03.yaml
Exists: True
path: /content/drive/MyDrive/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant


File: /content/Dataset_V1_Enhanced/well.v8i.yolov8/data.yaml
Exists: True
path: /content/drive/MyDrive/Dataset_V1_Enhanced/well.v8i.yolov8
train: /content/drive/MyDrive/Dataset_V1_Enhanced/well.v8i.yolov8/train/images
val: /content/drive/MyDrive/Dataset_V1/well.v8i.yolov8/valid/images
nc: 4
names: ['Inlet-pipe', 'fishes', 'school-of-fish', 'stone']



In [ ]:
# DIAT
DIAT_YAML="/content/Day23_DIAT.yaml"
with open(DIAT_YAML,"w") as f:
    f.write("""path: /content/Dataset_V1_Enhanced/DIATAquarium.v4i.yolov8
train: train/images
val: valid/images
nc: 11
names:
  0: Aquatic Plant
  1: Camera
  2: Conch-Shell
  3: Fish
  4: Fishes
  5: Pluco
  6: Shark
  7: Temperature Sensor
  8: Water-Pump
  9: Water-filter
  10: betta
""")
# Aquatic Plant
AP_YAML="/content/Day23_Aquatic_Plant.yaml"
with open(AP_YAML,"w") as f:
    f.write("""path: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 1
names:
  0: Aquatic_plant
""")

# Well
WELL_YAML="/content/Day23_Well.yaml"

with open(WELL_YAML,"w") as f:
    f.write("""path: /content/Dataset_V1_Enhanced/well.v8i.yolov8
train: train/images
val: valid/images
test: test/images
nc: 4
names:
  0: Inlet-pipe
  1: fishes
  2: school-of-fish
  3: stone
""")

print("Day23 YAML files created successfully.")
print(DIAT_YAML)
print(AP_YAML)
print(WELL_YAML)

Day23 YAML files created successfully.
/content/Day23_DIAT.yaml
/content/Day23_Aquatic_Plant.yaml
/content/Day23_Well.yaml


In [ ]:
from ultralytics import YOLO
import pandas as pd
# Aquatic Plant H03
AP_MODEL="/content/drive/MyDrive/Day22_Final_Model/H03_Training/Aquatic_Plant_H03/weights/best.pt"
ap_model=YOLO(AP_MODEL)

print("========== AQUATIC PLANT FINAL TEST ==========")
ap_test=ap_model.val(
    data="/content/Day23_Aquatic_Plant.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0
)
ap_result={
    "Dataset":"Aquatic Plant",
    "Configuration":"WB+CLAHE + YOLOv8n H03",
    "Split":"Test",
    "Precision":round(ap_test.box.mp*100,2),
    "Recall":round(ap_test.box.mr*100,2),
    "mAP50":round(ap_test.box.map50*100,2),
    "mAP50_95":round(ap_test.box.map*100,2)
}
print("\nAquatic Plant Final Test Result:")
print(ap_result)

========== AQUATIC PLANT FINAL TEST ==========
Ultralytics 8.4.158 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1080.4±356.6 MB/s, size: 50.2 KB)
val: Scanning /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/test/labels... 89 images, 38 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 89/89 2.3Kit/s 0.0s
val: New cache created: /content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.2it/s 2.7s
                   all         89         54       0.98      0.903      0.968      0.711
Speed: 5.9ms preprocess, 6.8ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/val-4

Aquatic Plant Final Test Result:
{'Dataset': 'Aquatic Plant', 'Configuration': 'WB+CLAHE + YOLOv8n H03', 'Spl

In [ ]:
from ultralytics import YOLO
WELL_MODEL="/content/drive/MyDrive/Day22_Final_Model/H04_Training/Well_H04/weights/best.pt"
well_model=YOLO(WELL_MODEL)
print("========== WELL FINAL TEST ==========")
well_test=well_model.val(
    data="/content/Day23_Well.yaml",
    split="test",
    imgsz=640,
    batch=16,
    device=0
)
well_result={
    "Dataset":"Well",
    "Configuration":"WB+CLAHE + YOLOv8n H04",
    "Split":"Test",
    "Precision":round(well_test.box.mp*100,2),
    "Recall":round(well_test.box.mr*100,2),
    "mAP50":round(well_test.box.map50*100,2),
    "mAP50_95":round(well_test.box.map*100,2)
}
print("\nWell Final Test Result:")
print(well_result)

========== WELL FINAL TEST ==========
Ultralytics 8.4.158 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1722.2±319.2 MB/s, size: 67.4 KB)
val: Scanning /content/Dataset_V1_Enhanced/well.v8i.yolov8/test/labels... 184 images, 21 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 184/184 2.1Kit/s 0.1s
val: New cache created: /content/Dataset_V1_Enhanced/well.v8i.yolov8/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.1it/s 5.8s
                   all        184       2111       0.43      0.375      0.393      0.162
                fishes        161       2065      0.646      0.746      0.699      0.262
        school-of-fish         23         43      0.644      0.378      0.481      0.223
                 stone          3          3          0          0          0   

In [ ]:
import os
import time
import pandas as pd
from ultralytics import YOLO
# Final model paths
DIAT_MODEL="/content/drive/MyDrive/Day22_Final_Model/DIAT_Baseline/weights/best.pt"
AP_MODEL="/content/drive/MyDrive/Day22_Final_Model/H03_Training/Aquatic_Plant_H03/weights/best.pt"
WELL_MODEL="/content/drive/MyDrive/Day22_Final_Model/H04_Training/Well_H04/weights/best.pt"
# Test/validation image folders
DIAT_IMAGES="/content/Dataset_V1/DIATAquarium.v4i.yolov8/valid/images"
AP_IMAGES="/content/Dataset_V1_Enhanced/Aquatic Plant.v2i.yolov8/test/images"
WELL_IMAGES="/content/Dataset_V1_Enhanced/well.v8i.yolov8/test/images"
def measure_inference(model_path,image_dir,n=100):
    model=YOLO(model_path)
    images=[
        os.path.join(image_dir,f)
        for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg",".jpeg",".png"))
    ][:n]
    print("Images selected:",len(images))
    # Warm-up
    model.predict(
        source=images[0],
        imgsz=640,
        device=0,
        verbose=False
    )
    start=time.time()
    for img in images:
        model.predict(
            source=img,
            imgsz=640,
            device=0,
            verbose=False
        )
    total_time=time.time()-start
    avg_ms=(total_time/len(images))*1000
    fps=1000/avg_ms
    return len(images),round(avg_ms,2),round(fps,2)
runtime_results=[]
print("========== DIAT ==========")
n,ms,fps=measure_inference(DIAT_MODEL,DIAT_IMAGES,100)
runtime_results.append({
    "Dataset":"DIAT",
    "Configuration":"Original + YOLOv8n",
    "Images_Measured":n,
    "Average_Inference_ms":ms,
    "FPS":fps
})
print("\n========== AQUATIC PLANT ==========")
n,ms,fps=measure_inference(AP_MODEL,AP_IMAGES,100)
runtime_results.append({
    "Dataset":"Aquatic Plant",
    "Configuration":"WB+CLAHE + YOLOv8n H03",
    "Images_Measured":n,
    "Average_Inference_ms":ms,
    "FPS":fps
})
print("\n========== WELL ==========")
n,ms,fps=measure_inference(WELL_MODEL,WELL_IMAGES,100)
runtime_results.append({
    "Dataset":"Well",
    "Configuration":"WB+CLAHE + YOLOv8n H04",
    "Images_Measured":n,
    "Average_Inference_ms":ms,
    "FPS":fps
})
runtime_df=pd.DataFrame(runtime_results)
print("\n========== DAY 23 RUNTIME RESULTS ==========")
display(runtime_df)

========== DIAT ==========
Images selected: 100

========== AQUATIC PLANT ==========
Images selected: 89

========== WELL ==========
Images selected: 100

========== DAY 23 RUNTIME RESULTS ==========


,Dataset,Configuration,Images_Measured,Average_Inference_ms,FPS
0,DIAT,Original + YOLOv8n,100,22.14,45.16
1,Aquatic Plant,WB+CLAHE + YOLOv8n H03,89,9.80,102.02
2,Well,WB+CLAHE + YOLOv8n H04,100,9.98,100.21


In [ ]:
import os
import pandas as pd
model_paths={
    "DIAT":DIAT_MODEL,
    "Aquatic Plant":AP_MODEL,
    "Well":WELL_MODEL
}
model_size_results=[]
for dataset,path in model_paths.items():
    size_mb=os.path.getsize(path)/(1024*1024)
    model_size_results.append({
        "Dataset":dataset,
        "Model":"YOLOv8n",
        "Model_Size_MB":round(size_mb,2),
        "Checkpoint":path
    })
model_size_df=pd.DataFrame(model_size_results)
print("========== DAY 23 MODEL SIZE ==========")
display(model_size_df)

========== DAY 23 MODEL SIZE ==========


,Dataset,Model,Model_Size_MB,Checkpoint
0,DIAT,YOLOv8n,5.95,/content/drive/MyDrive/Day22_Final_Model/DIAT_...
1,Aquatic Plant,YOLOv8n,5.96,/content/drive/MyDrive/Day22_Final_Model/H03_T...
2,Well,YOLOv8n,5.96,/content/drive/MyDrive/Day22_Final_Model/H04_T...


In [ ]:
import pandas as pd
comparison=pd.DataFrame([
    {
        "Dataset":"Aquatic Plant",
        "Baseline_Precision":95.69,
        "Final_Precision":97.99,
        "Baseline_Recall":94.44,
        "Final_Recall":90.26,
        "Baseline_mAP50":98.55,
        "Final_mAP50":96.75,
        "Baseline_mAP50_95":70.63,
        "Final_mAP50_95":71.12
    },
    {
        "Dataset":"Well",
        "Baseline_Precision":44.64,
        "Final_Precision":43.00,
        "Baseline_Recall":38.31,
        "Final_Recall":37.46,
        "Baseline_mAP50":42.20,
        "Final_mAP50":39.34,
        "Baseline_mAP50_95":18.46,
        "Final_mAP50_95":16.15
    }
])
comparison["Delta_Precision"]=(
    comparison["Final_Precision"]-
    comparison["Baseline_Precision"]
).round(2)

comparison["Delta_Recall"]=(
    comparison["Final_Recall"]-
    comparison["Baseline_Recall"]
).round(2)

comparison["Delta_mAP50"]=(
    comparison["Final_mAP50"]-
    comparison["Baseline_mAP50"]
).round(2)

comparison["Delta_mAP50_95"]=(
    comparison["Final_mAP50_95"]-
    comparison["Baseline_mAP50_95"]
).round(2)
display(comparison)

,Dataset,Baseline_Precision,Final_Precision,Baseline_Recall,Final_Recall,Baseline_mAP50,Final_mAP50,Baseline_mAP50_95,Final_mAP50_95,Delta_Precision,Delta_Recall,Delta_mAP50,Delta_mAP50_95
0,Aquatic Plant,95.69,97.99,94.44,90.26,98.55,96.75,70.63,71.12,2.30,-4.18,-1.80,0.49
1,Well,44.64,43.00,38.31,37.46,42.20,39.34,18.46,16.15,-1.64,-0.85,-2.86,-2.31


In [ ]:
import os
import pandas as pd
day23_folder="/content/drive/MyDrive/Day23_Final_Validation"
os.makedirs(day23_folder,exist_ok=True)
# Final validation results already measured
final_validation_df=pd.DataFrame([
    ["DIAT","Original + YOLOv8n","Validation",90.02,92.09,92.73,65.92],
    ["Aquatic Plant","WB+CLAHE + YOLOv8n H03","Test",97.99,90.26,96.75,71.12],
    ["Well","WB+CLAHE + YOLOv8n H04","Test",43.00,37.46,39.34,16.15]
],columns=[
    "Dataset","Configuration","Split",
    "Precision","Recall","mAP50","mAP50_95"
])
# Runtime results already measured
runtime_df=pd.DataFrame([
    ["DIAT","Original + YOLOv8n",100,22.14,45.16],
    ["Aquatic Plant","WB+CLAHE + YOLOv8n H03",89,9.80,102.02],
    ["Well","WB+CLAHE + YOLOv8n H04",100,9.98,100.21]
],columns=[
    "Dataset","Configuration","Images_Measured",
    "Average_Inference_ms","FPS"
])
# Model size results already measured
model_size_df=pd.DataFrame([
    ["DIAT","YOLOv8n",5.95],
    ["Aquatic Plant","YOLOv8n",5.96],
    ["Well","YOLOv8n",5.96]
],columns=[
    "Dataset","Model","Model_Size_MB"
])
# Baseline comparison already measured
comparison=pd.DataFrame([
    ["Aquatic Plant",95.69,97.99,94.44,90.26,98.55,96.75,70.63,71.12,2.30,-4.18,-1.80,0.49],
    ["Well",44.64,43.00,38.31,37.46,42.20,39.34,18.46,16.15,-1.64,-0.85,-2.86,-2.31]
],columns=[
    "Dataset",
    "Baseline_Precision","Final_Precision",
    "Baseline_Recall","Final_Recall",
    "Baseline_mAP50","Final_mAP50",
    "Baseline_mAP50_95","Final_mAP50_95",
    "Delta_Precision","Delta_Recall",
    "Delta_mAP50","Delta_mAP50_95"
])
# Save all four files
final_validation_df.to_csv(
    os.path.join(day23_folder,"final_validation_results.csv"),
    index=False
)
runtime_df.to_csv(
    os.path.join(day23_folder,"runtime_results.csv"),
    index=False
)
model_size_df.to_csv(
    os.path.join(day23_folder,"model_size_results.csv"),
    index=False
)
comparison.to_csv(
    os.path.join(day23_folder,"final_comparison.csv"),
    index=False
)
print("DAY 23 RESULTS SAVED")
print("="*50)
for file in os.listdir(day23_folder):
    print("✓",file)

DAY 23 RESULTS SAVED
✓ final_validation_results.csv
✓ runtime_results.csv
✓ model_size_results.csv
✓ final_comparison.csv


## Performance vs Complexity

Runtime and model size were measured for all three final configurations.

- **DIAT** (Original YOLOv8n): 22.14 ms/image,45.16 FPS
- **Aquatic Plant** (WB+CLAHE + H03): 9.80 ms/image,102.02 FPS
- **Well** (WB+CLAHE + H04): 9.98 ms/image,100.21 FPS

All three final models remain compact,with model sizes between 5.95 MB and 5.96 MB.

The runtime measurements represent YOLOv8n inference time on a Tesla T4 GPU. The separate preprocessing time for White Balance and CLAHE was not included in these inference measurements.

Therefore,the runtime results should be interpreted as model inference performance rather than the complete end-to-end preprocessing plus inference time.

## Final Findings

- DIAT used Original YOLOv8n because WB+CLAHE reduced its validation performance.
- Well's tuned configuration (H04) performed worse than the Original YOLOv8n baseline on every test metric: Precision -1.64pp,Recall -0.85pp,mAP@0.5 -2.86pp,and mAP@0.5:0.95 -2.31pp.

- Aquatic Plant's tuned configuration (H03) showed a trade-off on the test set: Precision (+2.30pp) and mAP@0.5:0.95 (+0.49pp) improved,while Recall (-4.18pp) and mAP@0.5 (-1.80pp) declined.

- This shows that selecting configurations using validation mAP@0.5:0.95 alone did not always predict test-set performance,particularly for the Well dataset.
- The effect of WB+CLAHE is dataset-dependent.
- All final models are approximately 5.95–5.96 MB.

## Limitations
- DIAT has no separate local test set.
- Source-level overlap exists in the supplied datasets.
- Well has strong class imbalance.
- WB+CLAHE was evaluated as a combined method.
- Water-Net and FUnIE-GAN were reviewed but not implemented.
- Tracking was not evaluated because video data was unavailable.
- Hyperparameter selection in Day 19 relied on validation mAP@0.5:0.95 as the primary metric,and this did not consistently generalize to the held-out test results.

## Conclusion

Final validation and test evaluation were completed using the frozen configurations.

The results show that WB+CLAHE does not consistently improve object detection across all datasets. Its effect depends on the dataset.

The final YOLOv8n models remained lightweight and provided fast inference on the Tesla T4 GPU.